# Karst Rainfall-Response Analysis — Google Colab Runner

Bu defter, `Karst-Rainfall-Response-Analysis` reposunu Google Colab üzerinde tek tıkla çalıştırmak için hazırlanmıştır:
1. Repo/Google Drive bağlantısı ve `requirements-colab.txt` kurulumu
2. Merkezi kozmetik ayarlarının (`config/plot_config.yaml`) düzenlenmesi
3. `main.py` ile analiz çalıştırma ve yüksek DPI statik grafiklerin görüntülenmesi
4. *(Opsiyonel)* Etkileşimli WebGL Streamlit arayüzünün Colab üzerinden başlatılması

In [ ]:
#@title 1. Repo / Google Drive Bağlantısı ve Kurulum
import os
from pathlib import Path

USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
REPO_URL = "https://github.com/<kullanici-adi>/Karst-Rainfall-Response-Analysis.git"  #@param {type:"string"}
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Karst-Rainfall-Response-Analysis"  #@param {type:"string"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path(DRIVE_PROJECT_DIR)
else:
    PROJECT_DIR = Path('/content/Karst-Rainfall-Response-Analysis')
    if not PROJECT_DIR.exists():
        !git clone "{REPO_URL}" "{PROJECT_DIR}"

os.chdir(PROJECT_DIR)
print("Aktif çalışma dizini:", Path.cwd())
!pip install -q -r requirements-colab.txt

In [ ]:
#@title 2. Merkezi Kozmetik Ayarlarını (config/plot_config.yaml) Düzenle
from pathlib import Path
import yaml

from src.plot_config import DEFAULT_CONFIG_PATH, load_plot_config

cfg = load_plot_config(DEFAULT_CONFIG_PATH)

# Örnek kozmetik özelleştirmeler (istediğiniz gibi değiştirebilirsiniz)
cfg['colors']['rainfall'] = '#0072B2'
cfg['colors']['discharge'] = '#D55E00'
cfg['colors']['discharge_peak'] = '#CC0000'
cfg['styles']['discharge_linewidth'] = 1.8
cfg['layout']['dpi'] = 200
cfg['layout']['save_formats'] = ['png', 'pdf']
cfg['layout']['show_temperature'] = True

CUSTOM_CONFIG_PATH = Path('outputs/colab_plot_config.yaml')
CUSTOM_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
CUSTOM_CONFIG_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
print(f'Özelleştirilmiş config kaydedildi: {CUSTOM_CONFIG_PATH}')

In [ ]:
#@title 3. Analizi Çalıştır (main.py) ve Grafikleri Göster
from pathlib import Path
from IPython.display import Image, display

# Kendi CSV dosyanızı kullanmak için yolunu yazın; boş bırakılırsa sentetik veri üretilir.
INPUT_CSV = ""  #@param {type:"string"}
MAX_LAG_DAYS = 14  #@param {type:"integer"}

cmd = f"python main.py --config outputs/colab_plot_config.yaml --max-lag {MAX_LAG_DAYS}"
if INPUT_CSV.strip():
    cmd += f' --input "{INPUT_CSV.strip()}"'

print('Çalıştırılan komut:', cmd)
!{cmd}

print('\nÜretilen PNG grafikler:')
for png_path in sorted(Path('outputs/figures').glob('*.png')):
    print(f'--- {png_path.name} ---')
    display(Image(filename=str(png_path)))

In [ ]:
#@title 4. (Opsiyonel) Etkileşimli Streamlit Arayüzünü Colab'de Başlat
# Cloudflare Quick Tunnel (kayıt/şifre gerektirmez) ile Streamlit arayüzüne erişim sağlar.
import re
import subprocess
import time

!wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared

streamlit_proc = subprocess.Popen(
    ['streamlit', 'run', 'app/streamlit_app.py', '--server.port=8501', '--server.headless=true'],
    stdout=open('/tmp/streamlit.log', 'w'),
    stderr=subprocess.STDOUT,
)
time.sleep(3)

tunnel_proc = subprocess.Popen(
    ['/tmp/cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print('Streamlit tüneli başlatılıyor...')
for line in tunnel_proc.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        print('Streamlit Arayüzü Hazır ->', match.group(0))
        break